In [1]:
import tensorflow as tf
import numpy as np
import os
import time

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from sklearn.metrics import classification_report


In [19]:
DATASET_PATH = r"D:\Project Jupyter\MODELLING\Dataset_baruPR"
TRAIN_DIR = os.path.join(DATASET_PATH, "train")
TEST_DIR  = os.path.join(DATASET_PATH, "test")


BASE_MODEL_PATH = r"D:\Project Jupyter\FINAL QUANTIZATION FIX\HASIL MODELBASE\mobilenetv3Large_jamur25COPYBARUFIXTF2PREPRO_tf"
TFLITE_INT8_PATH  = r"D:\Project Jupyter\mobilenetv3_BASE_LARGEint8FIXBARU.tflite"

# PARAMETER
IMG_SIZE = (224, 224)
BATCH_SIZE = 1                  # WAJIB 1 untuk INT8
CALIBRATION_SAMPLES = 100



In [21]:
pruned_fp32 = tf.keras.models.load_model(
    BASE_MODEL_PATH,
    compile=False
)

print("Model pruned FP32 berhasil")


Model pruned FP32 berhasil


In [22]:
def preprocess_with_pad(image):
    image = tf.image.resize_with_pad(
        image, IMG_SIZE[0], IMG_SIZE[1]
    )
    image = preprocess_input(image)
    return image


calib_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad
)

calib_data = calib_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

test_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad
)

test_data = test_gen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

class_names = list(test_data.class_indices.keys())


Found 2256 images belonging to 2 classes.
Found 232 images belonging to 2 classes.


In [23]:
def representative_dataset():
    calib_data.reset()
    for _ in range(CALIBRATION_SAMPLES):
        image, _ = next(calib_data)
        yield [image.astype(np.float32)]


In [30]:
converter = tf.lite.TFLiteConverter.from_saved_model(
    BASE_MODEL_PATH
)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

print("🔄 Melakukan Post-Training Quantization INT8")
tflite_int8_model = converter.convert()

with open(TFLITE_INT8_PATH, "wb") as f:
    f.write(tflite_int8_model)

model_size_mb = os.path.getsize(TFLITE_INT8_PATH) / (1024 * 1024)
print(f"✅ Model INT8 tersimpan ({model_size_mb:.2f} MB)")


🔄 Melakukan Post-Training Quantization INT8
✅ Model INT8 tersimpan (3.46 MB)


In [31]:
interpreter = tf.lite.Interpreter(model_path=TFLITE_INT8_PATH)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_scale, input_zero = input_details[0]["quantization"]
output_scale, output_zero = output_details[0]["quantization"]


In [32]:
def quantize_input(x):
    x_q = x / input_scale + input_zero
    return np.clip(x_q, -128, 127).astype(np.int8)

def dequantize_output(y):
    return (y.astype(np.float32) - output_zero) * output_scale


In [47]:
y_true, y_pred = [], []

test_data.reset()
start_time = time.time()

for i in range(len(test_data)):
    image, label = test_data[i]

    image_int8 = quantize_input(image)

    interpreter.set_tensor(
        input_details[0]["index"],
        image_int8
    )
    interpreter.invoke()

    output_int8 = interpreter.get_tensor(
        output_details[0]["index"]
    )
    output_fp32 = dequantize_output(output_int8)

    y_true.append(np.argmax(label))
    y_pred.append(np.argmax(output_fp32))

end_time = time.time()


In [48]:
total_time = end_time - start_time
avg_time = total_time / len(test_data)

print("\n=== HASIL PTQ INT8 (MODEL PRUNED) ===")
print(f"Total inference time : {total_time:.4f} detik")
print(f"Rata-rata per image  : {avg_time:.6f} detik")

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)



=== HASIL PTQ INT8 (MODEL PRUNED) ===
Total inference time : 637.8756 detik
Rata-rata per image  : 2.749464 detik

Classification Report:
              precision    recall  f1-score   support

      edible     0.7609    0.8750    0.8140       120
   poisonous     0.8404    0.7054    0.7670       112

    accuracy                         0.7931       232
   macro avg     0.8006    0.7902    0.7905       232
weighted avg     0.7993    0.7931    0.7913       232

